# Project Audible semiconductor pilot

Independent semiconductor and equipment sample. No real data or findings are
bundled. Run in its own WRDS directory. Resolve identities first, then review a
June 2024 example before setting FULL_HISTORY=True for 2004-2024.


In [ ]:
from pathlib import Path
import os, sys
project_root = Path.cwd().resolve()
if not (project_root / 'config' / 'audible.json').is_file():
    project_root = project_root.parent
if not (project_root / 'config' / 'audible.json').is_file():
    raise RuntimeError('Open this notebook inside the independent Audible project')
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print('Independent project:', project_root)


In [ ]:
import importlib
for name in ['pandas', 'numpy', 'wrds']:
    module = importlib.import_module(name)
    print(name, getattr(module, '__version__', 'available'))
# Use installed WRDS packages; do not install or upgrade packages here.


In [ ]:
import json
from src.common import config
from run_audible import load_settings, independent_data_root
settings = load_settings()
FULL_HISTORY = False
START = settings['start'] if FULL_HISTORY else settings['sample_start']
END = settings['end'] if FULL_HISTORY else settings['sample_end']
ANCHOR = settings['universe_asof']
DATA_ROOT = independent_data_root()
print('Labels:', config.load_universe())
print('Window:', START, END, 'Identity anchor:', ANCHOR)
print('Independent data root:', DATA_ROOT)


## Check identities before pulling history
This cell opens a real WRDS connection and checks columns and stock identity.
Use the normal WRDS authentication flow; never put a password in a cell.
Review all ten resolved IDs before continuing. The anchor does not make this a
point-in-time sample. Evidence is saved in a new independent directory.


In [ ]:
from run_audible import identity_check
if not config.WRDS_USERNAME:
    config.WRDS_USERNAME = input('WRDS username (not password): ').strip()
identity_folder, resolved = identity_check()
print('Identity evidence:', identity_folder)
display(resolved)


In [ ]:
from src.common.wrds_conn import connect
from src.stocks.pull import pull, raw_folder
if raw_folder(DATA_ROOT, START, END).exists():
    print('Raw directory exists. Review it; it will not be replaced.')
else:
    db = connect()
    try:
        print(pull(db, DATA_ROOT, START, END, config.load_universe(), ANCHOR))
    finally:
        db.close()


In [ ]:
from src.stocks.build import build
existing_output = DATA_ROOT / 'processed' / 'stocks' / f'{START}_{END}'
if existing_output.exists():
    raise FileExistsError(f'Processed output already exists: {existing_output}. Use a new audit run instead.')
output, report = build(DATA_ROOT, START, END)
print('Processed:', output)
print(json.dumps(report, indent=2))


In [ ]:
from src.stocks.history_audit import run
audit_output, summary = run(DATA_ROOT, START, END)
print('Audit:', audit_output)
print(json.dumps(summary, indent=2))
# Computational completion is not a clean-data claim. Review quality_status.


In [ ]:
from run_audible import summary_bundle
bundle = summary_bundle(audit_output)
print('Summary-only dashboard handoff:', bundle)
# Review the actual summaries and permitted coursework use before export.


## Before submission
Review actual semiconductor flags, quote sensitivity, returns and coverage.
Do not reuse original group counts as evidence for this sample. Keep market
rows and executed outputs on WRDS. Build the dashboard/memos from reviewed
semiconductor summaries, and clear outputs before sharing this source notebook.
